# Reading Topics Sceptically

**Computational Text Analysis Pack · Module 08 — Topic Modelling**

*Technique:* Choosing k; coherence; the topics that are artefacts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/05-unsupervised-structure/08c_reading_topics_sceptically.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

This is the notebook to keep open when you read someone else's topic model.

Four checks, in order of how often they are skipped:

1. **Is the number of topics defended, or was it 20 because everyone uses 20?**
2. **Do the topics survive a change of seed?**
3. **Is any topic just boilerplate, or a duplicate document set?**
4. **Would the argument change if a topic were named differently?**

Nothing here is exotic. It is all the humanities habit of asking where a category
came from, applied to a model.

## The tooling

**scikit-learn**'s `LatentDirichletAllocation` and `NMF`. **gensim** is faster
for large corpora and **MALLET** (Java) is what most published DH topic-model
work actually used — worth knowing when you read the literature.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

sub = df[df["language"] == "English"].reset_index(drop=True)
vec = CountVectorizer(token_pattern=r"[\w\u0300-\u036f]+", min_df=5, max_df=0.5, max_features=3000)
X = vec.fit_transform(sub["text"])
vocab = np.array(vec.get_feature_names_out())

## Check 1 — choosing k

Perplexity is what LDA optimises, so it is the obvious criterion. It is also
known to disagree with human judgement: models that score better on perplexity
often produce *less* interpretable topics.

**Topic coherence** — do a topic's top words actually co-occur in documents? —
tracks human judgement better. Here it is computed directly, no gensim required.

In [ ]:
def coherence(components, X, vocab, top_n=10):
    """UMass coherence: do a topic's top words appear in the same documents?"""
    binary = (X > 0).astype(np.int8)
    scores = []
    for comp in components:
        top = np.argsort(comp)[::-1][:top_n]
        total, pairs = 0.0, 0
        for a in range(1, len(top)):
            for b in range(a):
                d_ab = binary[:, top[a]].multiply(binary[:, top[b]]).sum()
                d_b = binary[:, top[b]].sum()
                if d_b:
                    total += np.log((d_ab + 1) / d_b)
                    pairs += 1
        scores.append(total / pairs if pairs else 0.0)
    return np.array(scores)

ks = [4, 6, 8, 10, 15, 20]
perp, coh = [], []
for k in ks:
    m = LatentDirichletAllocation(n_components=k, random_state=0,
                                  learning_method="batch", max_iter=20).fit(X)
    perp.append(m.perplexity(X))
    coh.append(coherence(m.components_, X, vocab).mean())
    print(f"k={k:>3}  perplexity {perp[-1]:>9.1f}   mean coherence {coh[-1]:>7.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(ks, perp, marker="o")
axes[0].set_xlabel("number of topics"); axes[0].set_ylabel("perplexity")
axes[0].set_title("perplexity — lower is 'better'")
axes[1].plot(ks, coh, marker="o", color="seagreen")
axes[1].set_xlabel("number of topics"); axes[1].set_ylabel("mean coherence")
axes[1].set_title("coherence — higher is more readable")
plt.tight_layout(); plt.show()
print("If those two curves point different ways, believe coherence — and then")
print("believe your reading of the topics over both of them.")

## Check 2 — seed stability

Fit the same model three times with different seeds and measure how many topics
recur. A topic that appears in one fit only is not a finding.

In [ ]:
K = 8
fits = [LatentDirichletAllocation(n_components=K, random_state=s,
                                  learning_method="batch", max_iter=20).fit(X)
        for s in (0, 1, 2)]
sets = [[set(vocab[np.argsort(c)[::-1][:10]]) for c in f.components_] for f in fits]

stable = 0
for i, s0 in enumerate(sets[0]):
    best = [max(len(s0 & s1) / 10 for s1 in other) for other in sets[1:]]
    tag = "stable" if min(best) >= 0.5 else "UNSTABLE"
    print(f"topic {i}: best overlap {min(best):.0%} across seeds  [{tag}]")
    stable += min(best) >= 0.5
print()
print(f"{stable} of {K} topics recur across all three fits.")

## Check 3 — is it boilerplate?

The crispest topic in a news corpus is very often a formula: a rights notice, a
sign-off, a syndication tag. It looks like a beautifully coherent theme.

In [ ]:
m = fits[0]
dt = m.transform(X)
for t in range(K):
    top_docs = np.argsort(dt[:, t])[::-1][:12]
    heads = sub.iloc[top_docs]["headline"].astype(str).str[:40]
    dupes = len(heads) - heads.nunique()
    words = ", ".join(vocab[np.argsort(m.components_[t])[::-1][:6]])
    flag = "  ← check for duplicates" if dupes >= 3 else ""
    print(f"topic {t}: {dupes:>2} repeated headlines in top 12   {words}{flag}")

### 🔧 Your turn 1

Take the topic with the most repeated headlines and print the full text of two of
its top documents.

Are they the same story republished? If so the topic is a duplicate-detection
result, and the fix is to deduplicate the corpus and refit — not to name the topic.

## Check 4 — would the argument survive a different name?

The final check needs no code. Write your topic's name, then write the most
sceptical alternative name someone could give the same word list.

If your argument holds under both, it is about the words. If it only holds under
your name, the argument is in the name.

In [ ]:
t = 0
words = vocab[np.argsort(fits[0].components_[t])[::-1][:12]]
print(f"topic {t} word list:")
print("   " + ", ".join(words))
print()
print("Your name for it            : ______________________")
print("The sceptic's name for it   : ______________________")
print()
print("Publish the word list. A reader who can see it can disagree with you,")
print("which is the difference between evidence and assertion.")

### 🔧 Your turn 2

Take a topic-model result from a published DH paper you have read and run the
four checks against what the paper reports.

How many can you even evaluate from what is printed? That number is usually zero
or one, and it is why reporting standards in this area are worth arguing about.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Repeated headlines among a topic's top documents almost always
means syndicated or duplicated text. Deduplicate on the text (or on a hash of its
first 200 characters), refit, and note in your methods how many documents you
removed. A model fitted on a corpus with 5% duplicates will devote a whole topic
to them.

**Your turn 2.** Most published topic-model results give a table of named topics
with their top words and nothing else — no k justification, no seed, no stability
check, no deduplication statement. You can usually evaluate check 4 alone. This is
not a reason to distrust the method; it is a reason for your own write-up to
include what theirs omits. A methods paragraph naming k and why, the seed, the
stability across seeds, and the deduplication step puts you ahead of most of the
literature and costs a paragraph.

</details>

## Where this stops

You can fit a topic model and defend it, or take one apart. Next: **Module 09 —
stylometry**, where the words that carry the signal are the ones every other
method throws away.